# Lesson 02 — MOG2: Mixture of Gaussians Background Model

```python
import cv2
import numpy as np
import matplotlib.pyplot as plt

cap = cv2.VideoCapture('sample_video.mp4')

# MOG2 builds a statistical model of the background
# history: how many frames to use for background model
# varThreshold: how different a pixel must be to be foreground
# detectShadows: detect and mark shadows (slower but useful)
mog2 = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True   # shadows marked as 127 (gray), foreground as 255
)

results = []
for i in range(30):
    ret, frame = cap.read()
    if not ret: break
    fg_mask = mog2.apply(frame)
    if i in [0, 5, 15, 29]:
        results.append((frame.copy(), fg_mask.copy(), i))

cap.release()

fig, axes = plt.subplots(len(results), 3, figsize=(18, len(results)*4))
for row, (frame, mask, idx) in enumerate(results):
    axes[row,0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    axes[row,0].set_title(f'Frame {idx}'); axes[row,0].axis('off')
    axes[row,1].imshow(mask, cmap='gray')
    axes[row,1].set_title('MOG2 mask (white=fg, gray=shadow)'); axes[row,1].axis('off')
    fg_only = cv2.bitwise_and(frame, frame, mask=(mask==255).astype(np.uint8)*255)
    axes[row,2].imshow(cv2.cvtColor(fg_only, cv2.COLOR_BGR2RGB))
    axes[row,2].set_title('Foreground only'); axes[row,2].axis('off')
plt.suptitle('MOG2 learns background over time — early frames noisier',fontsize=12)
plt.tight_layout(); plt.show()